In [8]:
import os

import bw2data as bd
import bw2io as bi
import pandas as pd
from bw2calc import LCA
from premise import *


In [9]:
# set brightway project:
bd.projects.set_current(name='premise-data-students-test')

In [10]:
# from relics import add_relics
# add_relics()

In [11]:
print(
    f"Existing BD databases in project {'premise-data-students-test'} :",
    bd.databases,
    "\n",
)

# Select premise database to conduct LCI
# premise_db =  bd.Database('ei_cutoff_3.10_image_SSP2-L_2050 2025-10-02')


Existing BD databases in project premise-data-students-test : Databases dictionary with 3 object(s):
	biosphere
	ecoinvent-3.10-cutoff
	ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040 2026-07-07 



**DEFINE DATABASES HERE** 

In [12]:
minerals = ['Aluminium','Chromium', 'Cobalt', 'Copper', 'Dysprosium', 'Gallium', 'Germanium', 'Graphite', 'Indium', 'Iridium', 'Lithium', 'Manganese', 'Neodymium', 'Nickel', 'Palladium', 'Platinum',  'Praseodymium', 'Selenium', 'Silicon', 'Silver', 'Tellurium', 'Vanadium', 'Zinc']
print(len(minerals))

# list of databases to access
# databases = [
#     "ecoinvent-3.10-cutoff",
#     "ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040 2025-10-20",
# ] 

databases = [
    "ecoinvent-3.10-cutoff",
    #  'ei_cutoff_3.10_image_SSP2-L_2050 2025-10-02',
    #  'ei_cutoff_3.10_image_SSP5-H_2050 2025-10-08',
    #  'ei_cutoff_3.10_image_SSP3-H_2050 2025-10-08',
    #  'ei_cutoff_3.10_remind_SSP2-PkBudg650_2050 2025-10-10',
    "ei_cutoff_3.10_tiam-ucl_SSP2-RCP19_2040 2026-07-07",
    # "ei_cutoff_3.10_tiam-ucl_SSP2-RCP26_2040 2026-07-07",
]


23


In [13]:
# Get list of technologies to fetch
technologies = pd.read_csv("data/lci_technologies.csv")

Get new activities for each database

In [14]:
def get_biosphere_flows(lca):
    """Extract biosphere flows and their amounts from LCA object."""
    rows, cols = lca.inventory.nonzero()
    data = []
    for row in set(rows):  # unique biosphere flows
        
        # print(lca.inventory[row, :])

        amount = lca.inventory[row, :].sum()  # total amount for this flow
        # Find the flow key that corresponds to this row
        flow_key = [k for k, v in lca.biosphere_dict.items() if v == row][0]
        flow_ds = bd.get_activity(flow_key)
        data.append({
            "flow name": flow_ds["name"],
            "categories": " / ".join(flow_ds.get("categories", [])),
            "unit": flow_ds["unit"],
            "amount": amount,
            "key": str(flow_ds.key)  # optional, unique identifier
        })
    return data

for database in databases: 
    db = bd.Database(database)

    save_path = f'results/{database}'

    if not os.path.exists(save_path):
        print(f"Creating directory for {database}")
        os.makedirs(save_path)

    for index, row in technologies.iterrows():
        activity = row['Activity']
        location = row['Location']

        # print(activity)

        file_save_name = f"{activity}_{location}.csv"
        files = os.listdir(f'results/{database}')

        if file_save_name not in files: 
            print(activity, location)
            activity_matches = [act for act in db if (activity == act['name'] and location == act['location'])]
            print("Number of activity matches", len(activity_matches))

            if len(activity_matches) > 0: 
                activity = activity_matches[0]
                lca = LCA({activity: 1})
                lca.lci()
                data = get_biosphere_flows(lca)


                # maybe should save all LCA data and then only put minerals in combined df
                df = pd.DataFrame(data)

                # filter for minerals only and for category "natural resources"
                # df = df[df['flow name'].str.contains('|'.join(minerals))]
                # df = df[df['categories'] == 'natural resource / in ground']

                # Save all biosphere data to a csv file
                df.to_csv(f"{save_path}/{file_save_name}", index=False)
                print(f"Completed LCA for activity: {activity['name']}, location: {activity['location']}")
            else: 
                print(f"{activity} {location} not found in {database}")
        else: 
            print(f"Activity data already retrieved for {activity}, {location} in {database}")
    
    files = os.listdir(f'results/{database}')
    dfs = []

    # create minerals only file with all activities 
    
    for idx, file in enumerate(files): 
        print(file)
        if (file != '.DS_Store') & (file != 'all_mineral_flows.csv'):
            df = pd.read_csv(f"{save_path}/{file}")

            df = df[df['flow name'].str.contains('|'.join(minerals))]
            df = df[df['categories'] == 'natural resource / in ground']

            df.set_index('flow name',inplace=True)
            df = df[['amount']].transpose()
            df['Activity'] = file.split(".csv")[0].split('_')[0]
            df['Location'] = file.split(".csv")[0].split('_')[1]
            
            dfs.append(df)

    df_all = pd.concat(dfs)

    # reorder columns 
    cols = list(df_all.columns)
    cols.remove('Activity')
    cols.remove('Location')
    new_cols = ['Activity', 'Location'] + cols


    df_all = df_all[new_cols]

    df_all.to_csv(f'results/{database}/all_mineral_flows.csv', index=False)

Activity data already retrieved for auxiliary heating unit production, electric, 5kW, RoW in ecoinvent-3.10-cutoff
Activity data already retrieved for battery production, Li-ion, LFP, rechargeable, prismatic, CN in ecoinvent-3.10-cutoff
Activity data already retrieved for battery production, Li-ion, NMC111, rechargeable, prismatic, CN in ecoinvent-3.10-cutoff
battery production, Li-ion, NMC532 GLO
Number of activity matches 0
battery production, Li-ion, NMC532 GLO not found in ecoinvent-3.10-cutoff
battery production, Li-ion, NMC622 GLO
Number of activity matches 0
battery production, Li-ion, NMC622 GLO not found in ecoinvent-3.10-cutoff
Activity data already retrieved for battery production, Li-ion, NMC811, rechargeable, prismatic, CN in ecoinvent-3.10-cutoff
Activity data already retrieved for electricity production, natural gas, combined cycle power plant, CH in ecoinvent-3.10-cutoff
Activity data already retrieved for electricity production, natural gas, conventional power plant, D

Combine results for all databases and standardize the units

In [15]:
all_dfs = []

for db in databases:
    df_temp = pd.read_csv(f'results/{db}/all_mineral_flows.csv')
    df_temp['database'] = db
    all_dfs.append(df_temp)

# Concatenate
df = pd.concat(all_dfs, ignore_index=True)


In [16]:
# need to merge on activity AND location
df = df.merge(technologies, how='left', on=['Activity', 'Location'])

In [17]:
# divide all mineral values by the multiplier to get equivalent units
df[minerals] = df[minerals].div(df.Multiplier, axis=0) 
df['Activity long'] = df['Activity'] + ' (' + df['Location'] + ')'

In [18]:
# fix column order
df = df[['Activity', 'database', 'Friendly name', 'Activity long', 'Unit', 'Multiplier', 'Location' ]+ minerals]

In [19]:
df.to_csv("results/all_dbs_standardised.csv")